# Download the LLM & prepare prompt for RAG

In [5]:
%pip install -U langchain-text-splitters langchain-community

Note: you may need to restart the kernel to use updated packages.


In [8]:
import os

print(os.getcwd())
print(os.listdir())

d:\Learn Python
['.git', '.git-rewrite', '.gitignore', '.ipynb_checkpoints', '100_PCA-ScikitLearn.ipynb', '101_PCA-ScikitLearn.ipynb', '102_ANN_TensorFlow_Keras.ipynb', '103_CNN_Model_TensorFlow_Keras.ipynb', '104_Basic_RNN.ipynb', '105_LSTM_Model_TensorFlow-Keras.ipynb', '106_Computer-vision-ImageProcessing-OpenCV.ipynb', '107_Object_Detection_YOLO_Algorithm.ipynb', '108_Image_SegmentationUNet_Architecture.ipynb', '109_Code Files', '10_Funtions.ipynb', '110_RAG-Embedding-Vector.ipynb', '111_RAG-Vector-Databases.ipynb', '112_RAG-Document-Loading-Data-Ingestion.ipynb', '113_RAG-pipeline.ipynb', '11_functionLocal.ipynb', '12-function-lambda.ipynb', '13_module.ipynb', '14_OOP.ipynb', '15_core_concepts.ipynb', '16_OOP-Class-Obj.ipynb', '17_OOP-constructor.ipynb', '18_OOP-Instance.ipynb', '19_OOP-Encapsulation.ipynb', '1_pythonintro.ipynb', '20_OOP-Inheritance.ipynb', '21_OOP_overriding.ipynb', '22_OOP-polymorphism.ipynb', '23_special-Methods.ipynb', '24_filehandling.ipynb', '25_mod-filehan

In [9]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = TextLoader("langchain_intro.txt")
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)

splits = text_splitter.split_documents(documents)

print(f"{len(splits)} chunks created.")

6 chunks created.


In [10]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Creating embeddings and indexing in FAISS...")

db = FAISS.from_documents(
    documents=splits,
    embedding=embedding_function
)

print("Vector Database created successfully!")
print(f"{len(splits)} vectors stored.")

C:\Users\HASNAIN\AppData\Local\Temp\ipykernel_3220\4191457796.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:03<00:00, 29.38it/s]


Creating embeddings and indexing in FAISS...
Vector Database created successfully!
6 vectors stored.


In [11]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

model_id = "Qwen/Qwen2-0.5B-Instruct"

print(f"Loading model: {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto", # use float16 if GPU available
    device_map="auto"   # Automatically use GPU if present, else CPU
)


# 2. Create HF Pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=128, # Limit length for speed
    temperature=0.1, # Lower temperature for less creativity/hallucination 
    do_sample=True # Enable sampling if needed, but low temp makes it deterministic
)

# 3. LangChain LLM Wrapper

raw_llm = HuggingFacePipeline(pipeline=pipe)

# 4. Define the Qwen Chat formatter
# This formats input as chat messages using Qwen's Templates.

def format_for_qwen(input_dict):
    messages = [
        { "role": "system", "content": """
You are a helpful AI assistant that answers questions strictly based on the provided context.
Do not use any external knowledge or make up information.
If the answer is not in the context, respond exactly with: "I don't know based on the provided context."
Always start your response with "Answer:" followed by the answer or the "I don't know" statement.
"""},
        {"role": "user", "content": f"""
Context:
{input_dict['context']}
Question: 
{input_dict["question"]}
"""}
    ]
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return formatted_prompt

# --- 5. Generation Function ---

def generate_with_qwen(formatted_prompt):
    # Use .invoke() instead of direct call
    response = raw_llm.invoke(formatted_prompt)

    # Extract generated text (after prompt)
    generated = response.split(formatted_prompt)[-1].strip()

    # Further clean: If it starts with "Answer:", extract after it
    if generated.startswith("Answer:"):
        generated = generated.split("Answer:", 1)[-1].strip()

    return generated


# --- 6. Create LLM Chain with Formatting ---

llm_with_format = (
    RunnableLambda(format_for_qwen)
    | RunnableLambda(generate_with_qwen)
    | StrOutputParser()
)

print("LLM setup complete!")



Loading model: Qwen/Qwen2-0.5B-Instruct...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2014.39it/s]


LLM setup complete!


# Building the RAG pipeline

#### Objective:


Based on "Building a RAG Pipeline" and "Step 4: Build Retriever + RetrievalQA Chain" slides. Combine the r
(Qwen) to answer a user question using the retrieved context.

In [12]:
from langchain_core.runnables import RunnablePassthrough
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

# --- 1. Define the Retriever

retriever = db.as_retriever(search_kwargs={"k": 2})

In [13]:
# --- 2. Create the RAG Chain ---

# Retrieval -> Format context & question -> LLM -> Output

rag_chain = (
    {
        "context": retriever | (lambda docs: "\n\n".join(
            doc.page_content for doc in docs
        )),
        "question": RunnablePassthrough()
    }
    | llm_with_format
)

In [14]:
# --- 3. Query Example ---

question = "What does RAG stand for?"

print(f"Question: {question}\n")

# Invoke the chain
response = rag_chain.invoke(question)

print("RAG Answer:")
print(response)

print("\n--- Context Used ---")

# Use .invoke() on the retriever to get documents
context_docs = retriever.invoke(question)

for doc in context_docs:
    print(doc.page_content)

Question: What does RAG stand for?



[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


RAG Answer:
RAG stands for Retrieval-Augmented Generation.

--- Context Used ---
RAG (Retrieval-Augmented Generation) is a popular technique used in LangChain.
Chains allow you to combine multiple components together.
